# Proyecto Centinela · Fase 2 — Rama C: Fusión Multimodal
### Serie temporal (GRU) + Imágenes (ResNet18) → Clasificación del régimen TRM

**Autores:** Estefanny Ruíz González · Miguel Alarcón Ojeda
**Curso:** Profundización I — Redes Neuronales / Deep Learning (EA-N-F-004)
**Maestría en Ciencia de Datos — Universidad Santo Tomás**

---

**Estrategia de fusión: concatenación tardía**
```
Ventana 30 días ──► GRU  ──► embedding  (64) ─┐
                                                ├─► concat (576) ──► Linear ──► 3 clases
Ventana 30 días ──► CNN  ──► embedding (512) ─┘
```
**Estudio de ablación:** Solo GRU vs. Solo CNN vs. Fusión completa.


## 0. Preparación del entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (confusion_matrix, classification_report,
                              ConfusionMatrixDisplay, accuracy_score, f1_score)

SEMILLA = 42
WINDOW  = 30
BATCH   = 32
EPOCAS  = 20
LR      = 0.001

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Dispositivo:", device)


## 1. Carga de datos y etiquetas

In [ ]:
from google.colab import files
subido = files.upload()  # subir trm_diaria_limpia.csv


In [ ]:
df = pd.read_csv("trm_diaria_limpia.csv", parse_dates=["fecha"])
df = df.sort_values("fecha").reset_index(drop=True)
df["retorno"] = df["trm"].pct_change() * 100
df = df.dropna(subset=["retorno"]).reset_index(drop=True)

retornos = df["retorno"].values
nombres_clases = ["Depreciación", "Estable", "Apreciación"]

etiquetas = []
for i in range(WINDOW, len(retornos)):
    acum = retornos[i-WINDOW:i].sum()
    if   acum >  1.0: etiquetas.append(0)
    elif acum < -1.0: etiquetas.append(2)
    else:             etiquetas.append(1)
etiquetas = np.array(etiquetas)

print("Ventanas:", len(etiquetas))
for c, n in enumerate(nombres_clases):
    cnt = (etiquetas==c).sum()
    print(f"  Clase {c} — {n}: {cnt} ({cnt/len(etiquetas)*100:.1f}%)")


## 2. Pre-cálculo de matrices de recurrencia

Las matrices se calculan **una sola vez** y se guardan en memoria.
Esto evita recalcularlas en cada batch durante el entrenamiento,
que era la causa del tiempo excesivo (>25 min por época).


In [ ]:
def matriz_recurrencia(serie, umbral=0.5):
    n = len(serie)
    mat = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(n):
            mat[i,j] = 1.0 if abs(serie[i]-serie[j]) < umbral else 0.0
    return mat

print("Pre-calculando matrices de recurrencia (tarda ~2 min)...")
todas_matrices = np.zeros((len(etiquetas), WINDOW, WINDOW), dtype=np.float32)
for i in range(len(etiquetas)):
    ventana = retornos[i:i+WINDOW]
    todas_matrices[i] = matriz_recurrencia(ventana)
    if (i+1) % 1000 == 0:
        print(f"  {i+1}/{len(etiquetas)} ventanas procesadas...")

print(f"Listo: {todas_matrices.shape}")
print(f"Memoria usada: {todas_matrices.nbytes/1e6:.1f} MB")


## 3. Partición y Dataset multimodal

In [ ]:
# Partición cronológica 70/15/15
indices = np.arange(len(etiquetas))
n = len(indices)
i_tr, i_va = int(n*0.70), int(n*0.85)
idx_train = indices[:i_tr]
idx_val   = indices[i_tr:i_va]
idx_test  = indices[i_va:]

print(f"Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}")


In [ ]:
transform_img = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

class DatasetMultimodal(Dataset):
    """Cada ejemplo entrega (serie, imagen, etiqueta).
    Las matrices están pre-calculadas para evitar recálculo en cada batch."""
    def __init__(self, matrices, retornos, etiquetas, indices):
        self.matrices  = matrices
        self.retornos  = retornos
        self.etiquetas = etiquetas
        self.indices   = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]

        # Entrada 1: serie para GRU — (30, 1)
        ventana = self.retornos[i:i+WINDOW]
        serie = torch.tensor(ventana, dtype=torch.float32).unsqueeze(-1)

        # Entrada 2: imagen pre-calculada — (3, 224, 224)
        mat = self.matrices[i]
        img = torch.tensor(mat).unsqueeze(0).repeat(3, 1, 1)
        img = transform_img(img)

        return serie, img, int(self.etiquetas[idx])

ds_train = DatasetMultimodal(todas_matrices, retornos, etiquetas, idx_train)
ds_val   = DatasetMultimodal(todas_matrices, retornos, etiquetas, idx_val)
ds_test  = DatasetMultimodal(todas_matrices, retornos, etiquetas, idx_test)

dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
dl_test  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print("DataLoaders listos")
print(f"Batches train: {len(dl_train)}")


## 4. Arquitecturas del estudio de ablación

Tres modelos con la misma tarea (clasificar el régimen de la TRM):

| Modelo | Entrada | Arquitectura |
|--------|---------|--------------|
| Solo GRU | Serie 30 retornos | GRU(64) → Linear(3) |
| Solo CNN | Imagen 30×30 | ResNet18 congelada → Linear(3) |
| Fusión | Serie + Imagen | GRU(64) + ResNet18(512) → Linear(576→128→3) |


In [ ]:
# ── Modelo 1: Solo GRU ──────────────────────────────────────────────────
class ModeloSoloGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(1, 64, 2, batch_first=True, dropout=0.2)
        self.fc  = nn.Linear(64, 3)

    def forward(self, serie, imagen=None):
        _, h = self.gru(serie)
        return self.fc(h[-1])

# ── Modelo 2: Solo CNN ──────────────────────────────────────────────────
class ModeloSoloCNN(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for p in resnet.parameters():
            p.requires_grad = False
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        self.fc  = nn.Linear(512, 3)

    def forward(self, serie=None, imagen=None):
        emb = self.cnn(imagen).squeeze(-1).squeeze(-1)
        return self.fc(emb)

# ── Modelo 3: Fusión GRU + CNN ──────────────────────────────────────────
class ModeloFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(1, 64, 2, batch_first=True, dropout=0.2)

        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for p in resnet.parameters():
            p.requires_grad = False
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])

        self.fusion = nn.Sequential(
            nn.Linear(64 + 512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3)
        )

    def forward(self, serie, imagen):
        _, h   = self.gru(serie)
        emb_s  = h[-1]                              # (batch, 64)
        emb_i  = self.cnn(imagen).squeeze(-1).squeeze(-1)  # (batch, 512)
        return self.fusion(torch.cat([emb_s, emb_i], dim=1))

# Verificar parámetros
for nombre, cls in [("Solo GRU", ModeloSoloGRU),
                    ("Solo CNN", ModeloSoloCNN),
                    ("Fusión",   ModeloFusion)]:
    m = cls()
    total   = sum(p.numel() for p in m.parameters())
    entrena = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{nombre:<12}: {total:>10,} total | {entrena:>8,} entrenables")


## 5. Función de entrenamiento unificada

In [ ]:
def entrenar(modelo, dl_tr, dl_va, nombre, epocas=EPOCAS, lr=LR):
    criterio = nn.CrossEntropyLoss()
    optim = torch.optim.Adam(
        [p for p in modelo.parameters() if p.requires_grad], lr=lr)

    hist_tr, hist_va = [], []
    mejor_val   = float("inf")
    mejor_pesos = None

    for epoca in range(epocas):
        modelo.train()
        loss_tr = 0.0
        for serie, img, label in dl_tr:
            serie = serie.to(device, non_blocking=True)
            img   = img.to(device,   non_blocking=True)
            label = label.to(device, non_blocking=True)
            optim.zero_grad()
            out  = modelo(serie, img)
            loss = criterio(out, label)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            optim.step()
            loss_tr += loss.item()
        loss_tr /= len(dl_tr)

        modelo.eval()
        loss_va = 0.0
        with torch.no_grad():
            for serie, img, label in dl_va:
                out = modelo(serie.to(device, non_blocking=True),
                             img.to(device,   non_blocking=True))
                loss_va += criterio(out, label.to(device)).item()
        loss_va /= len(dl_va)

        hist_tr.append(loss_tr)
        hist_va.append(loss_va)

        if loss_va < mejor_val:
            mejor_val   = loss_va
            mejor_pesos = {k: v.clone() for k,v in modelo.state_dict().items()}

        if epoca % 5 == 0:
            print(f"  Época {epoca:2d} | train: {loss_tr:.4f} | val: {loss_va:.4f}")

    modelo.load_state_dict(mejor_pesos)
    print(f"  Mejor val loss: {mejor_val:.4f}")
    return hist_tr, hist_va


## 6. Entrenamiento de los tres modelos

In [ ]:
resultados = {}

for nombre, cls in [("Solo GRU", ModeloSoloGRU),
                    ("Solo CNN", ModeloSoloCNN),
                    ("Fusión",   ModeloFusion)]:
    print(f"\n=== Entrenando {nombre} ===")
    torch.manual_seed(SEMILLA)
    modelo = cls().to(device)
    ht, hv = entrenar(modelo, dl_train, dl_val, nombre)
    resultados[nombre] = {"modelo": modelo, "ht": ht, "hv": hv}


## 7. Curvas de pérdida

In [ ]:
colores = {"Solo GRU": "#4C72B0", "Solo CNN": "#C44E52", "Fusión": "#55A868"}

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, nombre in zip(axes, ["Solo GRU", "Solo CNN", "Fusión"]):
    ht = resultados[nombre]["ht"]
    hv = resultados[nombre]["hv"]
    ax.plot(ht, color=colores[nombre], label="train")
    ax.plot(hv, color=colores[nombre], linestyle="--", label="val")
    ax.set_title(nombre)
    ax.set_xlabel("Época")
    ax.set_ylabel("Cross-Entropy Loss")
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle("Curvas de pérdida — estudio de ablación", fontsize=12)
plt.tight_layout(); plt.show()


## 8. Evaluación y estudio de ablación

In [ ]:
def evaluar(modelo, dl, nombre):
    modelo.eval()
    pred_all, real_all = [], []
    with torch.no_grad():
        for serie, img, label in dl:
            out  = modelo(serie.to(device), img.to(device))
            pred = out.argmax(dim=1).cpu().numpy()
            pred_all.extend(pred)
            real_all.extend(label.numpy())
    real_all = np.array(real_all)
    pred_all = np.array(pred_all)
    acc = accuracy_score(real_all, pred_all)
    f1  = f1_score(real_all, pred_all, average="macro", zero_division=0)
    print(f"\n=== {nombre} ===")
    print(f"Exactitud: {acc:.3f} | F1 macro: {f1:.3f}")
    print(classification_report(real_all, pred_all,
                                 target_names=nombres_clases,
                                 zero_division=0))
    return real_all, pred_all, acc, f1

metricas_ablacion = {}
for nombre in ["Solo GRU", "Solo CNN", "Fusión"]:
    r, p, acc, f1 = evaluar(resultados[nombre]["modelo"], dl_test, nombre)
    metricas_ablacion[nombre] = {"real": r, "pred": p, "acc": acc, "f1": f1}


In [ ]:
# Tabla resumen
print("\n=== TABLA DE ABLACIÓN ===")
print(f"{'Modelo':<14} {'Accuracy':>10} {'F1 macro':>10}")
print("-" * 36)
for nombre in ["Solo GRU", "Solo CNN", "Fusión"]:
    m = metricas_ablacion[nombre]
    print(f"{nombre:<14} {m['acc']:>10.3f} {m['f1']:>10.3f}")


In [ ]:
# Matrices de confusión
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, nombre in zip(axes, ["Solo GRU", "Solo CNN", "Fusión"]):
    cm = confusion_matrix(metricas_ablacion[nombre]["real"],
                          metricas_ablacion[nombre]["pred"])
    disp = ConfusionMatrixDisplay(cm, display_labels=nombres_clases)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    m = metricas_ablacion[nombre]
    ax.set_title(f"{nombre}\nAcc={m['acc']:.2f} | F1={m['f1']:.2f}")

plt.suptitle("Estudio de ablación — matrices de confusión (test)", fontsize=12)
plt.tight_layout(); plt.show()


## 9. Guardar modelos

In [ ]:
for nombre, key in [("Solo GRU","Solo GRU"),
                     ("Solo CNN","Solo CNN"),
                     ("Fusión",  "Fusión")]:
    fname = f"centinela_fase2_{key.lower().replace(' ','_').replace('ó','o')}.pt"
    torch.save(resultados[key]["modelo"].state_dict(), fname)
    print(f"Guardado: {fname}")


## 10. Resumen de la Rama C

**Arquitectura:** concatenación tardía.
- Embedding GRU (64 dims) + Embedding ResNet18 (512 dims) → Linear(576→128→3).
- Capas convolucionales de ResNet18 congeladas.

**Clave de alineación:** misma ventana de 30 días para ambas ramas.
El índice de ventana es el ancla común entre serie e imagen.

**Estudio de ablación:** permite cuantificar el aporte de cada modalidad.
Si la fusión supera a ambas ramas individuales, las modalidades son
complementarias. Si el GRU solo domina, la CNN con representaciones de
ImageNet no aporta información útil adicional para esta tarea.

**Limitación honesta:** serie y matrices de recurrencia provienen de la
misma fuente (TRM diaria). No son modalidades verdaderamente independientes.
Una segunda modalidad genuinamente independiente (imágenes satelitales,
datos de riesgo global) podría aportar más información complementaria.
